<a href="https://colab.research.google.com/github/brevfeed/pyspark/blob/main/notebooks/14-structured-streaming/01-sources-file-kafka-rate.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

*Part of the [BrevFeed PySpark notebooks](https://github.com/brevfeed/pyspark). Runs on real Spark 4.0 — for the in-browser course see [brevfeed.com/learn/pyspark](https://brevfeed.com/learn/pyspark).*


In [ ]:
# --- Colab setup: run me first (no-op if you already have Spark) --------------
# Installs a JDK + PySpark and fetches the sample data. ~1-2 min on a cold
# Colab runtime; instant on re-run. Safe to run locally too.
import os
import re
import subprocess
import sys
import urllib.request

DATA_URL = "https://raw.githubusercontent.com/brevfeed/pyspark/main/data"
DATA_DIR = "data"
IN_COLAB = "google.colab" in sys.modules


def _java_version():
    """Major version of the JDK on PATH, or 0 if there isn't one."""
    try:
        out = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
    except (FileNotFoundError, OSError):
        return 0
    m = re.search(r'version "(\d+)', out)
    return int(m.group(1)) if m else 0


# Spark 4 requires Java 17+. Colab ships an older JDK on some images.
if _java_version() < 17:
    if IN_COLAB:
        print("Installing OpenJDK 17 ...")
        subprocess.run(["apt-get", "-qq", "update"], check=False)
        subprocess.run(
            ["apt-get", "-qq", "install", "-y", "openjdk-17-jdk-headless"], check=True
        )
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"
    else:
        raise SystemExit(
            "Java 17+ is required for Spark 4. Install a JDK 17 and set JAVA_HOME."
        )

try:
    import pyspark  # noqa: F401
except ImportError:
    print("Installing PySpark ...")
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "pyspark==4.0.0"], check=True
    )

# Sample data lives in the repo; pull only what's missing.
os.makedirs(DATA_DIR, exist_ok=True)
for _f in ("orders.csv", "customers.csv", "events.jsonl", "sample_logs.txt"):
    _p = os.path.join(DATA_DIR, _f)
    if not os.path.exists(_p):
        urllib.request.urlretrieve(f"{DATA_URL}/{_f}", _p)


def spark_ui(spark=None):
    """Open the Spark UI — and make it reachable from Colab.

    The UI is served by the driver on the *runtime*, so on Colab plain
    http://localhost:4040 is not something your browser can reach. Colab can
    proxy a runtime port, so ask it for a real URL. Call this AFTER the
    SparkSession exists, or there is no server on the port yet.
    """
    port = 4040
    if spark is not None:  # the driver bumps the port if 4040 was taken
        try:
            port = int(spark.sparkContext.uiWebUrl.rsplit(":", 1)[1])
        except Exception:
            pass
    if not IN_COLAB:
        return f"http://localhost:{port}"
    from google.colab.output import eval_js

    url = eval_js(f"google.colab.kernel.proxyPort({port})")
    print(f"Spark UI: {url}  (open in a new tab)")
    return url


print("Ready. Sample data in ./data:", sorted(os.listdir(DATA_DIR)))
print("Spark UI: call spark_ui(spark) after creating the session.")


# 14.1 — Sources: file, Kafka, rate

`rate` for quick iteration, then a real `file` source demo (staged
writes + atomic rename into the watched directory, explicit schema
required). Kafka needs a broker we don't have locally -- shown as
real, correct, commented-out code (5.5's convention) plus a
schema-shape sanity check.

In [ ]:
import os, sys, shutil, time
os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import col

spark = (
    SparkSession.builder
    .appName("14.1-sources")
    .master("local[*]")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

## `rate` — synthetic, zero dependencies

In [ ]:
rate_df = spark.readStream.format("rate").option("rowsPerSecond", 5).load()
rate_df.printSchema()   # timestamp, value -- exactly the two columns 14.1 describes

## `file` — a directory as an unbounded table

Explicit schema (REQUIRED -- no inferSchema on a stream). Files are
staged elsewhere and renamed into the watched dir to simulate atomic
arrival, exactly as the lesson warns.

In [ ]:
from pyspark.sql.types import StructType, StructField, StringType, DoubleType

shutil.rmtree("stream_in", ignore_errors=True)
shutil.rmtree("stream_staging", ignore_errors=True)
os.makedirs("stream_in", exist_ok=True)
os.makedirs("stream_staging", exist_ok=True)

schema = StructType([
    StructField("customer_id", StringType()),
    StructField("amount", DoubleType()),
])

file_stream = (spark.readStream
               .schema(schema)
               .option("maxFilesPerTrigger", 1)
               .json("stream_in/"))

file_query = (file_stream.writeStream
              .format("memory").queryName("file_demo")
              .outputMode("append").start())

def emit_file(name, rows):
    """Write to staging, then RENAME into the watched dir -- atomic arrival."""
    staged = f"stream_staging/{name}"
    with open(staged, "w") as f:
        for r in rows:
            f.write(json.dumps(r) + "\n")
    os.replace(staged, f"stream_in/{name}")

import json
for i in range(3):
    emit_file(f"batch_{i}.json", [{"customer_id": f"c{i}", "amount": float(i * 10)}])
    time.sleep(2)

time.sleep(2)
spark.sql("SELECT * FROM file_demo ORDER BY customer_id").show()
file_query.stop()

## `kafka` — the production source (needs a real broker, not run here)

Real, correct code — commented, following 5.5's pattern. Uncomment
against an actual `kafka.bootstrap.servers` to run it for real.

In [ ]:
# kafka_df = (spark.readStream
#             .format("kafka")
#             .option("kafka.bootstrap.servers", "localhost:9092")
#             .option("subscribe", "orders-topic")
#             .option("startingOffsets", "latest")
#             .load())
#
# parsed = kafka_df.selectExpr("CAST(key AS STRING)", "CAST(value AS STRING)", "topic", "offset")

print("Kafka's streaming read schema is the SAME 7-column binary shape as 5.5's batch read:")
print("key, value: binary | topic, partition, offset, timestamp, timestampType")
print("-- the difference is entirely in continuous polling + offset tracking, not the shape")

In [ ]:
shutil.rmtree("stream_in", ignore_errors=True)
shutil.rmtree("stream_staging", ignore_errors=True)
spark.stop()

## Exercises

1. Change `maxFilesPerTrigger` to a higher number and re-run the file
   source demo with 6 files emitted quickly. Does the memory table
   still end up with all 6 rows? What changes about HOW they arrive
   (one micro-batch vs several)?
2. Deliberately write a file directly into `stream_in/` (skip the
   staging+rename step) while the query is running, mid-write (e.g.
   write it in two `f.write()` calls with a `time.sleep` between,
   without closing/renaming). What do you observe, and why does the
   lesson insist on atomic rename?
3. Set `rowsPerSecond` on the `rate` source to a very high number
   (e.g. 10000) and inspect `query.lastProgress` after a few seconds.
   What throttling-relevant fields do you see, and how would
   `maxOffsetsPerTrigger` play the same role for Kafka?
4. Try starting the file-source query a SECOND time (new query object,
   same `stream_in/` directory, no new files added). Does it reprocess
   the old files? What checkpoint-related concept from 14.6 do you
   think controls this?
5. `rate`'s `value` column is monotonically increasing. Design a
   `.select()` transformation that would make it look like a plausible
   fake "orders" stream (an id, an amount, a category) -- you'll reuse
   this pattern in later 14.x notebooks.

In [ ]:
# your exercise code here